<a href="https://colab.research.google.com/github/aymuos/endgame/blob/main/07_reward_oracle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Objective :Reward oracle for bandits simulator





Given: Features ↓ Incremental delivery duration

In [5]:
local = False

# Configure SIMD before importing NumPy/SciPy when running locally on AVX-512 hardware.
if local:
    import os
    os.environ.setdefault('MKL_ENABLE_INSTRUCTIONS', 'AVX512')
    os.environ.setdefault('OMP_NUM_THREADS', '1')  # joblib owns parallelism
    os.environ.setdefault('OPENBLAS_CORETYPE', 'SKYLAKEX')
    print('Local AVX-512 hints set (MKL_ENABLE_INSTRUCTIONS=AVX512, OPENBLAS_CORETYPE=SKYLAKEX)')

if not local:
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
! pip install lightgbm scikit-learn scikit-learn-intelex

In [7]:
if local :
# 1. Apply the AMD acceleration patch FIRST
    # 1. Trigger the AVX-512 level optimizations
    from sklearnex import patch_sklearn
    patch_sklearn()



In [8]:
from pathlib import Path
if local:
    root = Path('data')
    if not root.exists():
        root = Path('.data')
else:
    root = Path('/content/drive/MyDrive/ml/CORRECTEDv3')

DATA_PATHS = {
    'Chongqing': root / 'delivery_features_chongqing.parquet',
    'Shanghai': root / 'delivery_features_shanghai.parquet',
    'Hangzhou': root / 'delivery_features_hangzhou.parquet',
}

OUTPUT_DIR = root / 'causal_graphs' / 'pc_delivery'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [9]:
STATIC_FEATURES = [

    # Structural
    "pickup_destination_distance",
    "batch_size",
    "batch_rank_dispatch",
    "same_aoi_share_in_batch",
    "isolated_delivery",
    "distance_to_batch_centroid",
    "typecode_cb",

    # Operational
    "courier_eta_ewm",
    "gps_points",
    "speed_mean_15m",
    "speed_std_15m",
    "distance_travelled_15m",
    "coverage_ratio",
    "gps_gap_min",
    "idle_fraction",
    "is_trajectory_available",

    # Environment
    "WSI",
    "temperature_2m",
    "precipitation",
    "windspeed_10m",
    "spatial_congestion_daily",
    "spatial_congestion_norm",

    # Time
    "hour_sin",
    "hour_cos",
    "is_weekend",
    "is_holiday",
    "is_holiday_eve"
]




In [10]:
import pandas as pd

for city, path in DATA_PATHS.items():
    print(f"Processing {city} data...")
    try:
        df = pd.read_parquet(path)
        missing_features = [feature for feature in STATIC_FEATURES if feature not in df.columns]
        if not missing_features:
            print(f"  All STATIC_FEATURES are present in {city} dataframe.")
        else:
            print(f"  The following STATIC_FEATURES are missing in {city} dataframe: {missing_features}")
    except FileNotFoundError:
        print(f"  Error: File not found at {path}")
    except Exception as e:
        print(f"  An error occurred while processing {city} data: {e}")
print("\nFinished checking all dataframes.")

Processing Chongqing data...
  All STATIC_FEATURES are present in Chongqing dataframe.
Processing Shanghai data...
  All STATIC_FEATURES are present in Shanghai dataframe.
Processing Hangzhou data...
  All STATIC_FEATURES are present in Hangzhou dataframe.

Finished checking all dataframes.


In [16]:
DYNAMIC_FEATURES = [

    "dist_from_current",
    "remaining_orders",
    "batch_progress",
    "elapsed_route_time",
    "last_duration",
    "current_hour",
    "cumulative_distance"
]

In [41]:
import polars as pl

DYNAMIC_FEATURES = [
    "dist_from_current",
    "remaining_orders",
    "batch_progress",
    "elapsed_route_time",
    "last_duration",
    "current_hour",
    "cumulative_distance"
]

# true dispatch time is not available for all orders in LaDe, so for the first order
# we are going to use the receipt_time. This is a shortcoming and approximation.

# The objective is :

# The oracle predicts the duration before the action is taken.

# The state should therefore represent the system immediately before choosing the next delivery.

def compute_dynamic_features_polars(df: pl.DataFrame) -> pl.DataFrame:
    # Ensure datetime columns are in datetime format
    df = df.with_columns([
        pl.col('receipt_time').cast(pl.Datetime),
        pl.col('sign_time').cast(pl.Datetime)
    ])

    # Sort by batch_id and batch_rank_actual (0-indexed)
    sort_keys = ['batch_id', 'batch_rank_actual']
    df_sorted = df.sort(sort_keys)

    # 1. remaining_orders = total batch size - orders already delivered
    # At rank 0, we have 'batch_size' orders left to deliver.
    df_sorted = df_sorted.with_columns(
        (pl.col('batch_size').cast(pl.Int64) - pl.col('batch_rank_actual').cast(pl.Int64)).alias('remaining_orders')
    )

    # 2. batch_progress = batch_rank_actual / batch_size
    # Represents progress BEFORE the current delivery is completed.

    df_sorted = df_sorted.with_columns(
        (pl.col('batch_rank_actual').cast(pl.Float64) / pl.col('batch_size').cast(pl.Float64))
        .alias('batch_progress')
    )

    # Helper calculations for incremental duration
    duration_from_receipt = (
        (pl.col('sign_time') - pl.col('receipt_time')).dt.total_minutes()
        .fill_null(0.0)
        .clip(lower_bound=0)
    )

    duration_from_prev_sign = (
        (pl.col('sign_time') - pl.col('sign_time').shift(1).over('batch_id')).dt.total_minutes()
        .fill_null(0.0)
        .clip(lower_bound=0)
    )

    # 3. Incremental Duration
    df_sorted = df_sorted.with_columns(
        pl.when(pl.col('batch_rank_actual') == 0)
        .then(duration_from_receipt)
        .otherwise(duration_from_prev_sign)
        .alias('incremental_duration')
    )

    # 4. elapsed_route_time (Time elapsed BEFORE the current leg starts)
    df_sorted = df_sorted.with_columns(
        (pl.col('incremental_duration').cum_sum().over('batch_id') - pl.col('incremental_duration')).alias('elapsed_route_time')
    )

    # 5. Current Hour (Fractional hour at the start of this leg)
    df_sorted = df_sorted.with_columns(
        (pl.col('receipt_time') + pl.duration(minutes=pl.col('elapsed_route_time'))).alias('current_time')
    )
    df_sorted = df_sorted.with_columns(
        (pl.col('current_time').dt.hour() + pl.col('current_time').dt.minute().cast(pl.Float64)/60).alias('current_hour')
    )

    # 6. last_duration (Fixed order: shift -> over -> fill_null)
    df_sorted = df_sorted.with_columns(
        pl.col('incremental_duration')
        .shift(1)
        .over('batch_id')
        .fill_null(0.0)
        .alias('last_duration')
    )

    # 7. dist_from_current (Fixed order: shift -> over)
    df_sorted = df_sorted.with_columns([
        pl.col('poi_lng').shift(1).over('batch_id').alias('prev_lng'),
        pl.col('poi_lat').shift(1).over('batch_id').alias('prev_lat')
    ])

    df_sorted = df_sorted.with_columns(
        (
            ((pl.col('poi_lng') - pl.col('prev_lng'))**2 + (pl.col('poi_lat') - pl.col('prev_lat'))**2).sqrt()
        )
        .fill_null(pl.col('pickup_destination_distance'))
        .alias('dist_from_current')
    )

    # 8. cumulative_distance (Distance covered BEFORE this leg)
    df_sorted = df_sorted.with_columns(
        (pl.col('dist_from_current').cum_sum().over('batch_id') - pl.col('dist_from_current')).alias('cumulative_distance')
    )

    # Clean up temporary columns
    df_final = df_sorted.drop(['current_time', 'prev_lng', 'prev_lat', 'incremental_duration'])
    return df_final

processed_dfs = {}
for city, path in DATA_PATHS.items():
    print(f"\nProcessing {city} data...")
    try:
        df = pl.read_parquet(path)
        processed_df = compute_dynamic_features_polars(df)
        processed_dfs[city] = processed_df
        print(f"Finished {city}. 'Pre-action' logic applied")
    except Exception as e:
        print(f"Error processing {city}: {e}")


Processing Chongqing data...
Finished Chongqing. 'Pre-action' logic applied with comments restored.

Processing Shanghai data...
Finished Shanghai. 'Pre-action' logic applied with comments restored.

Processing Hangzhou data...
Finished Hangzhou. 'Pre-action' logic applied with comments restored.


In [36]:
# Using the Chongqing processed dataframe to verify incremental durations
city_sample = 'Chongqing'
if city_sample in processed_dfs:
    # We need to re-calculate incremental_duration briefly for display
    # since it was dropped at the end of the processing function
    df_check = processed_dfs[city_sample]

    # Pick a sample batch to inspect
    sample_bid = df_check.filter(pl.col('batch_size') > 1)['batch_id'][0]

    # Displaying the logic for the specific batch
    check_display = (df_check
        .filter(pl.col('batch_id') == sample_bid)
        .select([
            'batch_id',
            'batch_rank_actual',
            'last_duration',
            'elapsed_route_time'
        ])
    )
    print(f"Checking Batch: {sample_bid}")
    print(check_display)

Checking Batch: 0008c2b6a2314db8715301b7eeeebc5a__37e976ad4abb10da92c97c6b34f7f54f__318__1616055600
shape: (8, 4)
┌─────────────────────────────────┬───────────────────┬───────────────┬────────────────────┐
│ batch_id                        ┆ batch_rank_actual ┆ last_duration ┆ elapsed_route_time │
│ ---                             ┆ ---               ┆ ---           ┆ ---                │
│ str                             ┆ u32               ┆ f64           ┆ f64                │
╞═════════════════════════════════╪═══════════════════╪═══════════════╪════════════════════╡
│ 0008c2b6a2314db8715301b7eeeebc… ┆ 0                 ┆ 0.0           ┆ 0.0                │
│ 0008c2b6a2314db8715301b7eeeebc… ┆ 1                 ┆ 85.0          ┆ 85.0               │
│ 0008c2b6a2314db8715301b7eeeebc… ┆ 2                 ┆ 97.0          ┆ 182.0              │
│ 0008c2b6a2314db8715301b7eeeebc… ┆ 3                 ┆ 25.0          ┆ 207.0              │
│ 0008c2b6a2314db8715301b7eeeebc… ┆ 4            

In [38]:
# SANITY CHECK COLUMN - NEED NOT RUN
import polars as pl

# Re-running the core logic for Chongqing to inspect df_sorted before final cleanup
city_to_check = 'Chongqing'
raw_df = pl.read_parquet(DATA_PATHS[city_to_check])

# Standard preprocessing inside the function logic
df_check = raw_df.with_columns([
    pl.col('receipt_time').cast(pl.Datetime),
    pl.col('sign_time').cast(pl.Datetime)
]).sort(['batch_id', 'batch_rank_actual'])

duration_from_receipt = (
    (pl.col('sign_time') - pl.col('receipt_time')).dt.total_minutes()
    .fill_null(0.0)
    .clip(lower_bound=0)
)

duration_from_prev_sign = (
    (pl.col('sign_time') - pl.col('sign_time').shift(1).over('batch_id')).dt.total_minutes()
    .fill_null(0.0)
    .clip(lower_bound=0)
)

df_sorted_debug = df_check.with_columns(
    pl.when(pl.col('batch_rank_actual') == 0)
    .then(duration_from_receipt)
    .otherwise(duration_from_prev_sign)
    .alias('incremental_duration')
)

# Checking for nulls in incremental_duration and displaying the requested columns
null_count = df_sorted_debug.select(pl.col('incremental_duration').is_null().sum()).item()
print(f"Null count in incremental_duration: {null_count}")

print("First 50 rows of calculation check:")
display(df_sorted_debug.select([
    "batch_id",
    "batch_rank_actual",
    "incremental_duration"
]).head(50))

Null count in incremental_duration: 0
First 50 rows of calculation check:


batch_id,batch_rank_actual,incremental_duration
str,u32,f64
"""0008c2b6a2314db8715301b7eeeebc…",0,85.0
"""0008c2b6a2314db8715301b7eeeebc…",1,97.0
"""0008c2b6a2314db8715301b7eeeebc…",2,25.0
"""0008c2b6a2314db8715301b7eeeebc…",3,27.0
"""0008c2b6a2314db8715301b7eeeebc…",4,27.0
…,…,…
"""0008c2b6a2314db8715301b7eeeebc…",0,1.0
"""0008c2b6a2314db8715301b7eeeebc…",0,39.0
"""0008c2b6a2314db8715301b7eeeebc…",1,21.0


In [35]:
# Verification check for 0-indexed rank logic NO NEED TO RUN FOR NORMAL FLOW
city_sample = 'Chongqing'
if city_sample in processed_dfs:
    df_check = processed_dfs[city_sample]
    # Look at a specific batch with more than 1 order to verify ranges
    multi_order_batches = df_check.filter(pl.col('batch_size') > 1).select('batch_id').unique().head(1)
    if not multi_order_batches.is_empty():
        bid = multi_order_batches.item()
        print(f"Verification for Batch ID: {bid}")
        print(df_check.filter(pl.col('batch_id') == bid).select([
            'batch_id', 'batch_size', 'batch_rank_actual', 'remaining_orders', 'batch_progress'
        ]))

Verification for Batch ID: 0008c2b6a2314db8715301b7eeeebc5a__37e976ad4abb10da92c97c6b34f7f54f__318__1616055600
shape: (8, 5)
┌─────────────────────────────┬────────────┬───────────────────┬──────────────────┬────────────────┐
│ batch_id                    ┆ batch_size ┆ batch_rank_actual ┆ remaining_orders ┆ batch_progress │
│ ---                         ┆ ---        ┆ ---               ┆ ---              ┆ ---            │
│ str                         ┆ u32        ┆ u32               ┆ i64              ┆ f64            │
╞═════════════════════════════╪════════════╪═══════════════════╪══════════════════╪════════════════╡
│ 0008c2b6a2314db8715301b7eee ┆ 8          ┆ 0                 ┆ 7                ┆ 0.0            │
│ ebc…                        ┆            ┆                   ┆                  ┆                │
│ 0008c2b6a2314db8715301b7eee ┆ 8          ┆ 1                 ┆ 6                ┆ 0.142857       │
│ ebc…                        ┆            ┆                   ┆   

In [28]:
import polars as pl
# Inspect columns to find the batch identifier
for city, path in DATA_PATHS.items():
    df_cols = pl.read_parquet(path).columns
    print(f"{city} columns: {df_cols}")
    break # Just check one to identify the column name

Chongqing columns: ['order_id', 'from_dipan_id', 'delivery_user_id', 'poi_lng', 'poi_lat', 'aoi_id', 'typecode', 'receipt_time', 'receipt_lng', 'receipt_lat', 'sign_time', 'ds', 'eta_mins', 'pickup_destination_distance', 'batch_size', 'isolated_delivery', 'same_aoi_share_in_batch', 'batch_rank_dispatch', 'batch_rank_actual', 'batch_id', 'distance_to_batch_centroid', 'gps_points', 'speed_mean_15m', 'speed_std_15m', 'distance_travelled_15m', 'idle_fraction', 'coverage_ratio', 'ds_right', 'last_gps_time', 'last_x', 'last_y', 'gps_gap_min', 'courier_eta_ewm', 'remaining_haul_distance', 'is_trajectory_available', 'hour', 'weekday', 'receipt_date', 'hour_sin', 'hour_cos', 'is_weekend', 'is_holiday', 'is_holiday_eve', 'typecode_grouped_missing', 'typecode_grouped_other', 'typecode_grouped_type_1', 'typecode_grouped_type_2', 'typecode_cb', 'grid_x', 'grid_y', 'time_window', 'spatial_congestion_daily', 'spatial_congestion_norm', 'city_right', 'datetime', 'temperature_2m', 'precipitation', 'rain

## Sanity Check:

For the dynamic features , we created we want to check the following crieterias ,

For every batch we should verify:

- incremental_duration >= 0
- elapsed_route_time is monotonic increasing
- last_duration equals the previous row's incremental_duration
- remaining_orders decreases by exactly one at each step
- batch_progress increases monotonically from 0 toward 1
- cumulative_distance never decreases

In [42]:
def validate_batch_logic(df: pl.DataFrame, city_name: str):
    print(f"\n>>> Deep Sanity Check: {city_name} <<<")

    # We need to temporarily recreate incremental_duration for the check
    # as it was dropped in the final processing step
    df_check = df.with_columns([
        pl.when(pl.col('batch_rank_actual') == 0)
        .then((pl.col('sign_time') - pl.col('receipt_time')).dt.total_minutes())
        .otherwise((pl.col('sign_time') - pl.col('sign_time').shift(1).over('batch_id')).dt.total_minutes())
        .alias('_inc_dur')
    ])

    # 1. incremental_duration >= 0
    neg_dur = df_check.filter(pl.col('_inc_dur') < 0).height

    # 2. elapsed_route_time is monotonic increasing
    # 6. cumulative_distance never decreases
    # We check if the difference between current and previous is negative
    temporal_anomalies = df_check.filter(
        (pl.col('elapsed_route_time') < pl.col('elapsed_route_time').shift(1).over('batch_id')) |
        (pl.col('cumulative_distance') < pl.col('cumulative_distance').shift(1).over('batch_id'))
    ).height

    # 3. last_duration equals the previous row's incremental_duration
    # Note: rank 0 has no previous, so we skip it
    last_dur_mismatch = df_check.filter(
        (pl.col('batch_rank_actual') > 0) &
        (pl.col('last_duration') != pl.col('_inc_dur').shift(1).over('batch_id'))
    ).height

    # 4. remaining_orders decreases by exactly one
    rank_logic_err = df_check.filter(
        (pl.col('batch_rank_actual') > 0) &
        (pl.col('remaining_orders') != (pl.col('remaining_orders').shift(1).over('batch_id') - 1))
    ).height

    # 5. batch_progress increases monotonically
    progress_err = df_check.filter(
        (pl.col('batch_rank_actual') > 0) &
        (pl.col('batch_progress') <= pl.col('batch_progress').shift(1).over('batch_id'))
    ).height

    print(f"  - Total Records Checked: {df.height}")
    print(f"  - Negative Durations: {neg_dur}")
    print(f"  - Temporal/Distance Monotonicity Violations: {temporal_anomalies}")
    print(f"  - 'last_duration' Mismatches: {last_dur_mismatch}")
    print(f"  - Rank/Remaining Order Inconsistencies: {rank_logic_err}")
    print(f"  - Progress Monotonicity Violations: {progress_err}")

for city, df_processed in processed_dfs.items():
    validate_batch_logic(df_processed, city)


>>> Deep Sanity Check: Chongqing <<<
  - Total Records Checked: 25335
  - Negative Durations: 0
  - Temporal/Distance Monotonicity Violations: 0
  - 'last_duration' Mismatches: 0
  - Rank/Remaining Order Inconsistencies: 0
  - Progress Monotonicity Violations: 0

>>> Deep Sanity Check: Shanghai <<<
  - Total Records Checked: 34480
  - Negative Durations: 0
  - Temporal/Distance Monotonicity Violations: 0
  - 'last_duration' Mismatches: 0
  - Rank/Remaining Order Inconsistencies: 0
  - Progress Monotonicity Violations: 0

>>> Deep Sanity Check: Hangzhou <<<
  - Total Records Checked: 40348
  - Negative Durations: 0
  - Temporal/Distance Monotonicity Violations: 0
  - 'last_duration' Mismatches: 0
  - Rank/Remaining Order Inconsistencies: 0
  - Progress Monotonicity Violations: 0
